In [33]:
from pathlib import Path
import pandas as pd

current_folder = Path.cwd()

if (current_folder / "data").exists():
    project_root = current_folder
else:
    project_root = current_folder.parent

raw_file = (
    project_root
    / "data"
    / "raw"
    / "clinvar_download_20260712_233035.tsv"
)

df = pd.read_csv(raw_file, sep="\t")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\Dell - i5 11th Gen\\Desktop\\clinvar-gnomad-vus-analyzer\\notebooks\\data\\raw\\clinvar_download_20260712_233035.tsv'

In [ ]:
df.shape

(19651, 16)

In [ ]:
df.columns.tolist()

['Variation',
 'Genes',
 'Type',
 'Condition',
 'Classification',
 'Review status',
 'Molecular consequence',
 'Protein change',
 'Germline date last evaluated',
 'Somatic clinical impact date last evaluated',
 'Oncogenicity date last evaluated',
 'GRCh37 Location',
 'GRCh38 Location',
 'Variation ID',
 'VCV accession',
 'rs_ID']

In [ ]:
atm_missense_vus = df[
    (df["Genes"] == "ATM") &
    (df["Classification"] == "G: Uncertain significance") &
    (df["Molecular consequence"] == "missense variant")
].copy()

atm_missense_vus.shape

(4564, 16)

In [ ]:
atm_missense_vus = atm_missense_vus.reset_index(drop=True)

atm_missense_vus.head(10)

,Variation,Genes,Type,Condition,Classification,Review status,Molecular consequence,Protein change,Germline date last evaluated,Somatic clinical impact date last evaluated,Oncogenicity date last evaluated,GRCh37 Location,GRCh38 Location,Variation ID,VCV accession,rs_ID
0,NM_000051.4(ATM):c.4A>C (p.Ser2Arg),ATM,single nucleotide variant,Hereditary cancer-predisposing syndrome,G: Uncertain significance,"criteria provided, single submitter",missense variant,S2R,2025/08/26,NaN,NaN,11:108098355,11:108227628,4170131,VCV004170131,rs4279232
1,NM_000051.4(ATM):c.4A>G (p.Ser2Gly),ATM,single nucleotide variant,"Hereditary cancer-predisposing syndrome, Famil...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,S2G,2024/02/19,NaN,NaN,11:108098355,11:108227628,640846,VCV000640846,rs639367
2,NM_000051.4(ATM):c.5G>C (p.Ser2Thr),ATM,single nucleotide variant,not specified,G: Uncertain significance,"criteria provided, single submitter",missense variant,S2T,2014/06/23,NaN,NaN,11:108098356,11:108227629,181943,VCV000181943,rs180377
3,NM_000051.4(ATM):c.6T>G (p.Ser2Arg),ATM,single nucleotide variant,not specified,G: Uncertain significance,"criteria provided, single submitter",missense variant,S2R,2019/12/09,NaN,NaN,11:108098357,11:108227630,1337452,VCV001337452,rs1328461
4,NM_000051.4(ATM):c.6T>A (p.Ser2Arg),ATM,single nucleotide variant,"Hereditary cancer-predisposing syndrome, Ataxi...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,S2R,2025/11/01,NaN,NaN,11:108098357,11:108227630,922075,VCV000922075,rs911284
5,NM_000051.4(ATM):c.8T>C (p.Leu3Pro),ATM,single nucleotide variant,"Hereditary cancer-predisposing syndrome, Ataxi...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,L3P,2023/01/28,NaN,NaN,11:108098359,11:108227632,453759,VCV000453759,rs460775
6,NM_000051.4(ATM):c.10G>T (p.Val4Leu),ATM,single nucleotide variant,"Hereditary cancer-predisposing syndrome, Ataxi...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,V4L,2025/01/12,NaN,NaN,11:108098361,11:108227634,481224,VCV000481224,rs475312
7,NM_000051.4(ATM):c.13C>T (p.Leu5Phe),ATM,single nucleotide variant,Ataxia-telangiectasia syndrome,G: Uncertain significance,"criteria provided, single submitter",missense variant,L5F,2022/03/10,NaN,NaN,11:108098364,11:108227637,2108203,VCV002108203,rs2159617
8,NM_000051.4(ATM):c.13C>G (p.Leu5Val),ATM,single nucleotide variant,"Hereditary cancer-predisposing syndrome, not p...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,L5V,2025/04/29,NaN,NaN,11:108098364,11:108227637,1059654,VCV001059654,rs1046907
9,NM_000051.4(ATM):c.14T>G (p.Leu5Arg),ATM,single nucleotide variant,"Hereditary cancer-predisposing syndrome, not s...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,L5R,2023/04/26,NaN,NaN,11:108098365,11:108227638,1773992,VCV001773992,rs1827098


In [ ]:
print("Rows and columns:", atm_missense_vus.shape)

assert (atm_missense_vus["Genes"] == "ATM").all()
assert (atm_missense_vus["Classification"] == "G: Uncertain significance").all()
assert (atm_missense_vus["Molecular consequence"] == "missense variant").all()

print("All rows passed the filters!")

Rows and columns: (4564, 16)
All rows passed the filters!


In [ ]:
from pathlib import Path

processed_folder = Path("data/processed")
processed_folder.mkdir(parents=True, exist_ok=True)

atm_missense_vus.to_csv(
    processed_folder / "atm_missense_vus_all.csv",
    index=False
)

In [ ]:
eligible_variants = atm_missense_vus[
    (
        atm_missense_vus["rs_ID"]
        .astype("string")
        .str.fullmatch(r"rs\d+", na=False)
    )
    &
    (atm_missense_vus["GRCh38 Location"].notna())
    &
    (
        atm_missense_vus["Review status"]
        .str.contains("criteria provided", case=False, na=False)
    )
].copy()

eligible_variants.shape

(4559, 16)

In [ ]:
selected_variants = eligible_variants.sample(
    n=20,
    random_state=42
).reset_index(drop=True)

selected_variants[
    [
        "Variation",
        "Protein change",
        "Review status",
        "GRCh38 Location",
        "Variation ID",
        "VCV accession",
        "rs_ID"
    ]
]

,Variation,Protein change,Review status,GRCh38 Location,Variation ID,VCV accession,rs_ID
0,NM_000051.4(ATM):c.584C>A (p.Thr195Asn),T195N,"criteria provided, single submitter",11:108244040,2831360,VCV002831360,rs3000960
1,NM_000051.4(ATM):c.4336G>A (p.Val1446Ile),V1446I,"criteria provided, single submitter",11:108289701,847462,VCV000847462,rs837751
2,NM_000051.4(ATM):c.37C>G (p.Arg13Gly),R13G,"criteria provided, multiple submitters, no con...",11:108227661,824233,VCV000824233,rs810136
3,NM_000051.4(ATM):c.4477C>G (p.Leu1493Val),L1493V,"criteria provided, multiple submitters, no con...",11:108292659,644374,VCV000644374,rs639519
4,NM_000051.4(ATM):c.182T>C (p.Phe61Ser),F61S,"criteria provided, multiple submitters, no con...",11:108227885,186585,VCV000186585,rs183082
5,NM_000051.4(ATM):c.2989G>C (p.Val997Leu),V997L,"criteria provided, multiple submitters, no con...",11:108271318,1045945,VCV001045945,rs1029966
6,NM_000051.4(ATM):c.4996G>A (p.Glu1666Lys),E1666K,"criteria provided, single submitter",11:108297373,2842052,VCV002842052,rs3000589
7,NM_000051.4(ATM):c.2977C>T (p.His993Tyr),H993Y,"criteria provided, single submitter",11:108271306,2700158,VCV002700158,rs2859010
8,NM_000051.4(ATM):c.1776C>A (p.Ser592Arg),S592R,"criteria provided, single submitter",11:108252005,482586,VCV000482586,rs475732
9,NM_000051.4(ATM):c.5232G>C (p.Lys1744Asn),K1744N,"criteria provided, multiple submitters, no con...",11:108301702,961265,VCV000961265,rs947242


In [34]:
selected_variants.to_csv(
    "data/processed/atm_selected_vus_20.csv",
    index=False
)